In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# เริ่มต้นใช้งาน Sampler

งานหลักของ Sampler คือการสุ่มตัวอย่าง output register จากการรัน quantum circuits หนึ่งรายการหรือมากกว่า [Dynamic circuits](/guides/execute-dynamic-circuits) และ parameterized circuits ถูกรับเป็น input (ถ้าส่ง parametrized circuits ต้องระบุค่า parameter ด้วย) Sampler ยังรองรับ dynamical decoupling และ twirling ในตัวสำหรับ [error suppression](/guides/error-mitigation-and-suppression-techniques)

ขั้นตอนในหัวข้อนี้อธิบายวิธีตั้งค่า Sampler สำรวจตัวเลือกที่ใช้กำหนดค่า และเรียกใช้ในโปรแกรม


<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

## ขั้นตอนการใช้ Sampler primitive
### 1. เริ่มต้นบัญชี
เนื่องจาก Qiskit Runtime เป็นบริการที่จัดการให้ คุณต้องเริ่มต้นบัญชีก่อน จากนั้นจึงเลือก QPU ที่ต้องการใช้คำนวณค่าความคาดหวัง

ทำตามขั้นตอนใน [ตั้งค่าบัญชี IBM Cloud](/guides/cloud-setup) หากยังไม่มีบัญชีที่ตั้งค่าไว้

:::note[Fractional gates]

เพื่อใช้ [fractional gates](/guides/fractional-gates) ที่รองรับใหม่ ให้ตั้งค่า `use_fractional_gates=True` เมื่อขอ backend จาก `QiskitRuntimeService` instance ตัวอย่างเช่น:

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

นี่เป็นคุณสมบัติทดลองและอาจเปลี่ยนแปลงในอนาคต

:::

In [2]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(127, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

### 2. สร้าง circuit
คุณต้องมีอย่างน้อยหนึ่ง circuit เป็น input ให้ Sampler primitive

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 3036), ('sx', 1769), ('cz', 378), ('measure', 127), ('barrier', 1)])


Circuit และ observable ต้องถูกแปลงเพื่อใช้เฉพาะคำสั่งที่ QPU รองรับ (เรียกว่า *instruction set architecture (ISA)* circuits) ใช้ transpiler เพื่อทำสิ่งนี้

In [4]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

### 4. Invoke Sampler and get results

Next, invoke the `run()` method to generate the output. The circuit and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: dab7lb6rrl7c7386gjgg


>>> Job Status: QUEUED


In [6]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(
    f"First ten results for the 'meas' output register: "
    f"{pub_result.data.meas.get_bitstrings()[:10]}"
)

First ten results for the 'meas' output register: ['1110001000100100100111011000001001000101010000001100010010110011001100110111100000000110001000011001101000110100101110101010011', '1110101100010101010001010111111011000110011010100101001100110101010010101010101110001010100100000000011110101101011000100000100', '1100110011101000110000000100110111101010110001100010111100001101100001010100111001101010101110000101010001001101010101001000100', '1011001001011000000101010111111010011101001111010100001011110011000101110101011001100110011011110011101010000111001111110110011', '1110111011011111101011110101110101010110011001100110101111100100011101100011011100101100101001001111110101111010101110111111011', '0101001110001110010001010100010111011001000000011110000000000001110000010010101101011001000100000001011011101111010010110111110', '1101101001000100010000110001000101010010101011000011000011001010000111100110010101010110100110011010101100001011100110001111011', '1000110011111111100010101010101

### 4. เรียกใช้ Sampler และรับผลลัพธ์
จากนั้น เรียกใช้เมธอด `run()` เพื่อสร้าง output โดย circuit และชุดค่า parameter ที่เป็นตัวเลือก (optional) ถูกป้อนเป็น *primitive unified bloc* (PUB) tuples